# Quick Draw - PyTorch MobileNet with Multi-GPU & MPS Support

## Key Optimizations:
- **Multi-GPU**: Dual T4 support via `DataParallel` / `DistributedDataParallel`
- **MPS Support**: Native Apple Silicon GPU acceleration
- **Mixed Precision**: `torch.cuda.amp` for faster training
- **Efficient DataLoader**: Multi-worker prefetching
- **70k samples per class**
- **OneCycleLR**: Fast convergence scheduler

## Part 1: Setup & Device Detection

In [ ]:
# Install required packages if needed
# !pip install timm  # For optimized MobileNet implementations

In [ ]:
%matplotlib inline
import os
import sys
import json
import gc
import datetime as dt
import platform
from pathlib import Path
from typing import Tuple, List, Optional, Dict, Any
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, IterableDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts
from torch.cuda.amp import GradScaler, autocast

# Try to import timm for optimized models
try:
    import timm
    HAS_TIMM = True
    print("✓ timm available for optimized models")
except ImportError:
    HAS_TIMM = False
    from torchvision.models import mobilenet_v2
    print("⚠ timm not available, using torchvision MobileNetV2")

plt.rcParams['figure.figsize'] = [16, 10]
plt.rcParams['font.size'] = 14

START_TIME = dt.datetime.now()
print(f"PyTorch version: {torch.__version__}")
print(f"Started at: {START_TIME}")

In [ ]:
# ============================================
# DEVICE DETECTION & SETUP
# ============================================

@dataclass
class DeviceConfig:
    """Device configuration container."""
    device: torch.device
    device_type: str  # 'cuda', 'mps', 'cpu'
    num_devices: int
    use_amp: bool  # Automatic Mixed Precision
    device_names: List[str]


def setup_device() -> DeviceConfig:
    """
    Detect and configure the best available compute device.
    
    Priority: CUDA (multi-GPU) > MPS (Apple Silicon) > CPU
    """
    
    # Check CUDA (NVIDIA GPUs)
    if torch.cuda.is_available():
        num_gpus = torch.cuda.device_count()
        device_names = [torch.cuda.get_device_name(i) for i in range(num_gpus)]
        
        print(f"✓ CUDA available: {num_gpus} GPU(s)")
        for i, name in enumerate(device_names):
            mem = torch.cuda.get_device_properties(i).total_memory / 1e9
            print(f"  GPU {i}: {name} ({mem:.1f} GB)")
        
        # Enable TF32 for Ampere GPUs (A100, RTX 30xx)
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
        
        return DeviceConfig(
            device=torch.device('cuda'),
            device_type='cuda',
            num_devices=num_gpus,
            use_amp=True,
            device_names=device_names
        )
    
    # Check MPS (Apple Silicon)
    if torch.backends.mps.is_available():
        print("✓ MPS (Apple Silicon) available")
        print(f"  Device: {platform.processor()}")
        
        return DeviceConfig(
            device=torch.device('mps'),
            device_type='mps',
            num_devices=1,
            use_amp=False,  # MPS doesn't support AMP yet
            device_names=['Apple Silicon GPU']
        )
    
    # Fallback to CPU
    print("⚠ No GPU detected, using CPU")
    return DeviceConfig(
        device=torch.device('cpu'),
        device_type='cpu',
        num_devices=0,
        use_amp=False,
        device_names=['CPU']
    )


# Setup device
DEVICE_CFG = setup_device()
DEVICE = DEVICE_CFG.device
print(f"\n→ Using: {DEVICE_CFG.device_type.upper()} (AMP: {DEVICE_CFG.use_amp})")

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

@dataclass
class Config:
    """Training configuration."""
    # Paths
    input_dir: str = '/kaggle/input/quickdraw-doodle-recognition/'
    
    data_dir: str = '/kaggle/working/shuffled_data/'  # <-- UPDATE THIS TOO


    # Data format - NEW!
    data_format: str = 'parquet'  # 'parquet' or 'csv'
    file_prefix: str = 'train_k'  # 'train_k' for parquet, 'train_data_part_' for csv

    valid_split: float = 0.02  # 2% for validation
    
    # Data
    base_size: int = 256
    num_csvs: int = 100
    num_classes: int = 340
    samples_per_class: int = 70000
    
    # Model
    image_size: int = 64
    line_width: int = 6
    
    # Training - will be scaled based on GPU count
    batch_size_per_gpu: int = 340  # Per GPU
    num_workers: int = 4  # DataLoader workers
    epochs: int = 20
    lr: float = 3e-3 
    weight_decay: float = 1e-4
    
    # Scheduler
    warmup_epochs: int = 2
    
    # Reproducibility
    seed: int = 1987
    
    @property
    def global_batch_size(self) -> int:
        return self.batch_size_per_gpu * max(1, DEVICE_CFG.num_devices)


CFG = Config()

# Set seeds
def seed_everything(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)

print(f"Configuration:")
print(f"  Samples per class: {CFG.samples_per_class:,}")
print(f"  Total samples: ~{CFG.samples_per_class * CFG.num_classes:,}")
print(f"  Batch size per GPU: {CFG.batch_size_per_gpu}")
print(f"  Global batch size: {CFG.global_batch_size}")
print(f"  Learning rate: {CFG.lr}")

In [ ]:
# ============================================
# ADD MISSING IMPORT
# ============================================
import glob

# ============================================
# DATA FILE DISCOVERY (PARQUET + CSV SUPPORT)
# ============================================

def discover_data_files(cfg: Config) -> Tuple[List[int], int]:
    """
    Dynamically discover available data files.
    
    Returns:
        Tuple of (sorted list of file indices, total count)
    """
    # Build pattern based on format
    if cfg.data_format == 'parquet':
        pattern = os.path.join(cfg.data_dir, f'{cfg.file_prefix}*.parquet')
    else:
        pattern = os.path.join(cfg.data_dir, f'{cfg.file_prefix}*.csv')
    
    files = glob.glob(pattern)
    
    if not files:
        print(f"⚠ No data files found matching: {pattern}")
        # Show what's actually in the directory
        all_files = glob.glob(os.path.join(cfg.data_dir, '*'))
        if all_files:
            print(f"  Files found in {cfg.data_dir}:")
            for f in all_files[:5]:
                print(f"    - {os.path.basename(f)}")
            if len(all_files) > 5:
                print(f"    ... and {len(all_files) - 5} more")
        return [], 0
    
    # Extract indices from filenames
    indices = []
    ext = '.parquet' if cfg.data_format == 'parquet' else '.csv'
    
    for f in files:
        basename = os.path.basename(f)
        try:
            # Handle both 'train_k0.parquet' and 'train_data_part_0.csv' patterns
            idx_str = basename.replace(cfg.file_prefix, '').replace(ext, '')
            idx = int(idx_str)
            indices.append(idx)
        except ValueError:
            continue
    
    indices = sorted(indices)
    print(f"✓ Found {len(indices)} {cfg.data_format} files (indices: {indices[0]} to {indices[-1]})")
    
    return indices, len(indices)


def split_train_valid_indices(all_indices: List[int], valid_ratio: float = 0.02) -> Tuple[List[int], List[int]]:
    """
    Split file indices into train and validation sets.
    
    Args:
        all_indices: List of available file indices
        valid_ratio: Fraction of files for validation
    
    Returns:
        (train_indices, valid_indices)
    """
    if len(all_indices) == 0:
        raise ValueError("No data files available!")
    
    n_valid = max(1, int(len(all_indices) * valid_ratio))
    
    # Use last files for validation (they're already shuffled)
    train_indices = all_indices[:-n_valid]
    valid_indices = all_indices[-n_valid:]
    
    print(f"  Train files: {len(train_indices)} (indices {train_indices[0]} to {train_indices[-1]})")
    print(f"  Valid files: {len(valid_indices)} (indices {valid_indices})")
    
    return train_indices, valid_indices

## Part 2: Data Preparation

In [ ]:
# ============================================
# HELPER FUNCTIONS
# ============================================

def f2cat(filename: str) -> str:
    """Extract category name from filename."""
    return filename.split('.')[0]


def list_all_categories(input_dir: str) -> List[str]:
    """List all category names from training data."""
    files = os.listdir(os.path.join(input_dir, 'train_simplified'))
    return sorted([f2cat(f) for f in files], key=str.lower)


def draw_strokes(raw_strokes: List, size: int = 256, 
                 lw: int = 6, time_color: bool = True) -> np.ndarray:
    """
    Render strokes to grayscale image with temporal coloring.
    
    Args:
        raw_strokes: List of strokes [[x_coords], [y_coords]]
        size: Output image size
        lw: Line width
        time_color: Earlier strokes are darker
    
    Returns:
        Grayscale image as numpy array (0-255)
    """
    img = np.zeros((CFG.base_size, CFG.base_size), dtype=np.uint8)
    
    for t, stroke in enumerate(raw_strokes):
        color = 255 - min(t, 10) * 13 if time_color else 255
        for i in range(len(stroke[0]) - 1):
            cv2.line(
                img,
                (stroke[0][i], stroke[1][i]),
                (stroke[0][i + 1], stroke[1][i + 1]),
                color, lw
            )
    
    if size != CFG.base_size:
        img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    
    return img

In [ ]:
# ============================================
# FAST DATA PREPARATION (KAGGLE)
# ============================================

import os
import gc
import datetime as dt
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

print("✓ Using Parquet format (10-50x faster than gzip)")

# Fix paths for Kaggle
DATA_DIR = '/kaggle/working/shuffled_data/'
INPUT_DIR = '/kaggle/input/quickdraw-doodle-recognition/'


def fast_shuffle(df, seed):
    rng = np.random.RandomState(seed)
    idx = np.arange(len(df), dtype=np.int32)
    rng.shuffle(idx)
    return df.iloc[idx].reset_index(drop=True)


def process_file(args):
    filepath, seed = args
    if not os.path.exists(filepath):
        return
    df = pd.read_csv(filepath)
    df = fast_shuffle(df, seed)
    df.to_parquet(filepath.replace('.csv', '.parquet'), compression='snappy')
    os.remove(filepath)


def create_shuffled_csvs(cfg, force=False):
    data_dir = '/kaggle/working/shuffled_data/'
    input_dir = '/kaggle/input/quickdraw-doodle-recognition/'
    
    os.makedirs(data_dir, exist_ok=True)
    
    if os.path.exists(os.path.join(data_dir, 'train_k0.parquet')) and not force:
        print("✓ Parquet data exists")
        return
    
    print(f"\n{'='*50}")
    print(f"Samples/class: {cfg.samples_per_class:,}")
    print(f"Output: {data_dir}")
    print(f"{'='*50}\n")
    
    start = dt.datetime.now()
    
    # Get categories
    files = os.listdir(os.path.join(input_dir, 'train_simplified'))
    categories = sorted([f.split('.')[0] for f in files], key=str.lower)
    print(f"Found {len(categories)} categories\n")
    
    # Phase 1: Read and split
    print("[1/2] Reading...")
    for y, cat in enumerate(tqdm(categories)):
        df = pd.read_csv(
            os.path.join(input_dir, 'train_simplified', f'{cat}.csv'),
            nrows=cfg.samples_per_class,
            usecols=['key_id', 'drawing']
        )
        df['y'] = y
        df['cv'] = (df.key_id // 10**7) % cfg.num_csvs
        
        for k in range(cfg.num_csvs):
            chunk = df[df.cv == k][['drawing', 'y']]
            if len(chunk) == 0:
                continue
            path = os.path.join(data_dir, f'train_k{k}.csv')
            chunk.to_csv(path, mode='w' if y == 0 else 'a',
                        header=(y == 0), index=False)
        
        del df
        if y % 50 == 0:
            gc.collect()
    
    # Phase 2: Parallel shuffle + Parquet
    print("\n[2/2] Shuffling (parallel)...")
    args = [(os.path.join(data_dir, f'train_k{k}.csv'), k)
            for k in range(cfg.num_csvs)]
    
    with ThreadPoolExecutor(max_workers=4) as ex:
        list(tqdm(ex.map(process_file, args), total=len(args)))
    
    # Update config to use new path
    cfg.data_dir = data_dir
    
    elapsed = (dt.datetime.now() - start).total_seconds()
    print(f"\n✓ Done in {elapsed/60:.1f} min")


# Update config path and run
CFG.data_dir = '/kaggle/working/shuffled_data/'
create_shuffled_csvs(CFG, force=True)

In [ ]:
# ============================================
# DATA VISUALIZATION
# ============================================

import json
import random

# Get category names
files = os.listdir(os.path.join(INPUT_DIR, 'train_simplified'))
categories = sorted([f.split('.')[0] for f in files], key=str.lower)
id2cat = {i: cat for i, cat in enumerate(categories)}

# Load sample data
df = pd.read_parquet(os.path.join(DATA_DIR, 'train_k0.parquet'))
print(f"Sample file shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nClass distribution in this file:")
print(df['y'].value_counts().describe())


# Drawing function
def draw_strokes(raw_strokes, size=128, lw=3, time_color=True):
    img = np.zeros((256, 256), np.uint8)
    for t, stroke in enumerate(raw_strokes):
        color = 255 - min(t, 10) * 13 if time_color else 255
        for i in range(len(stroke[0]) - 1):
            cv2.line(img, (stroke[0][i], stroke[1][i]),
                    (stroke[0][i + 1], stroke[1][i + 1]), color, lw)
    return cv2.resize(img, (size, size))


# ============================================
# 1. RANDOM SAMPLES GRID
# ============================================
print("\n" + "="*50)
print("RANDOM SAMPLES")
print("="*50)

fig, axes = plt.subplots(6, 8, figsize=(16, 12))
fig.suptitle('Random Samples from Dataset', fontsize=16)

samples = df.sample(48)
for idx, (ax, (_, row)) in enumerate(zip(axes.flat, samples.iterrows())):
    strokes = json.loads(row['drawing'])
    img = draw_strokes(strokes)
    ax.imshow(img, cmap='gray_r')
    ax.set_title(id2cat[row['y']][:12], fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()


# ============================================
# 2. SAMPLES BY CATEGORY
# ============================================
print("\n" + "="*50)
print("SAMPLES BY CATEGORY")
print("="*50)

# Pick 8 random categories
random_cats = random.sample(range(len(categories)), 8)

fig, axes = plt.subplots(8, 6, figsize=(12, 16))
fig.suptitle('6 Samples per Category', fontsize=16)

for row_idx, cat_id in enumerate(random_cats):
    cat_samples = df[df['y'] == cat_id].head(6)
    for col_idx, (_, row) in enumerate(cat_samples.iterrows()):
        ax = axes[row_idx, col_idx]
        strokes = json.loads(row['drawing'])
        img = draw_strokes(strokes)
        ax.imshow(img, cmap='gray_r')
        if col_idx == 0:
            ax.set_ylabel(id2cat[cat_id][:15], fontsize=9)
        ax.axis('off')

plt.tight_layout()
plt.show()


# ============================================
# 3. CLASS DISTRIBUTION
# ============================================
print("\n" + "="*50)
print("CLASS DISTRIBUTION")
print("="*50)

# Load all parquet files to get full distribution
all_counts = []
for k in range(100):
    path = os.path.join(DATA_DIR, f'train_k{k}.parquet')
    if os.path.exists(path):
        temp_df = pd.read_parquet(path, columns=['y'])
        all_counts.append(temp_df['y'].value_counts())

total_counts = pd.concat(all_counts).groupby(level=0).sum().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
axes[0].hist(total_counts.values, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Samples per Class')
axes[0].set_ylabel('Number of Classes')
axes[0].set_title('Distribution of Samples per Class')
axes[0].axvline(total_counts.mean(), color='r', linestyle='--', 
                label=f'Mean: {total_counts.mean():,.0f}')
axes[0].legend()

# Top/Bottom classes
top_10 = total_counts.nlargest(10)
bottom_10 = total_counts.nsmallest(10)

y_pos = range(10)
axes[1].barh(y_pos, top_10.values, alpha=0.7, label='Top 10')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels([id2cat[i][:20] for i in top_10.index])
axes[1].set_xlabel('Sample Count')
axes[1].set_title('Top 10 Classes by Sample Count')

plt.tight_layout()
plt.show()

print(f"\nTotal samples: {total_counts.sum():,}")
print(f"Classes: {len(total_counts)}")
print(f"Min samples/class: {total_counts.min():,}")
print(f"Max samples/class: {total_counts.max():,}")
print(f"Mean samples/class: {total_counts.mean():,.0f}")


# ============================================
# 4. STROKE STATISTICS
# ============================================
print("\n" + "="*50)
print("STROKE STATISTICS")
print("="*50)

# Analyze strokes from sample
stroke_counts = []
point_counts = []

for _, row in df.head(5000).iterrows():
    strokes = json.loads(row['drawing'])
    stroke_counts.append(len(strokes))
    point_counts.append(sum(len(s[0]) for s in strokes))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(stroke_counts, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Number of Strokes')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Strokes per Drawing (mean: {np.mean(stroke_counts):.1f})')

axes[1].hist(point_counts, bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Number of Points')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Points per Drawing (mean: {np.mean(point_counts):.1f})')

plt.tight_layout()
plt.show()

print(f"Strokes: min={min(stroke_counts)}, max={max(stroke_counts)}, mean={np.mean(stroke_counts):.1f}")
print(f"Points: min={min(point_counts)}, max={max(point_counts)}, mean={np.mean(point_counts):.1f}")

## Part 3: PyTorch Dataset & DataLoader

In [ ]:
# ============================================
# DATASET CLASSES (PARQUET SUPPORT)
# ============================================

class QuickDrawDataset(Dataset):
    """In-memory dataset for validation (loads all data upfront)."""
    
    def __init__(self, cfg: Config, file_indices: List[int], max_samples: Optional[int] = None):
        self.cfg = cfg
        self.size = cfg.image_size
        self.lw = cfg.line_width
        
        # Load data from specified files
        dfs = []
        loaded_files = []
        
        print(f"Loading data from {len(file_indices)} file(s)...")
        
        for idx in file_indices:
            # Support both parquet and csv
            if cfg.data_format == 'parquet':
                filepath = os.path.join(cfg.data_dir, f'{cfg.file_prefix}{idx}.parquet')
            else:
                filepath = os.path.join(cfg.data_dir, f'{cfg.file_prefix}{idx}.csv')
            
            if os.path.exists(filepath):
                print(f"  ✓ Loading: {filepath}")
                if cfg.data_format == 'parquet':
                    dfs.append(pd.read_parquet(filepath))
                else:
                    dfs.append(pd.read_csv(filepath))
                loaded_files.append(idx)
            else:
                print(f"  ⚠ File not found: {filepath}")
        
        if len(dfs) == 0:
            # Try to discover what files actually exist
            existing = glob.glob(os.path.join(cfg.data_dir, '*'))
            raise FileNotFoundError(
                f"No data files found!\n"
                f"  Looked for: {cfg.file_prefix}*.{cfg.data_format}\n"
                f"  In directory: {cfg.data_dir}\n"
                f"  Files in dir: {existing[:10]}..."
            )
        
        self.df = pd.concat(dfs, ignore_index=True)
        print(f"  Loaded {len(self.df):,} samples from {len(loaded_files)} files")
        
        if max_samples and len(self.df) > max_samples:
            self.df = self.df.sample(n=max_samples, random_state=42).reset_index(drop=True)
            print(f"  Sampled down to {max_samples:,}")
    
    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        row = self.df.iloc[idx]
        
        strokes = json.loads(row['drawing'])
        img = draw_strokes(strokes, size=self.size, lw=self.lw)
        
        img = torch.from_numpy(img).float().unsqueeze(0)
        img = (img / 127.5) - 1.0
        
        return img, int(row['y'])


class QuickDrawStreamingDataset(IterableDataset):
    """Streaming dataset for training (memory efficient)."""
    
    def __init__(self, cfg: Config, file_indices: List[int]):
        self.cfg = cfg
        self.file_indices = list(file_indices)  # Make a copy
        self.size = cfg.image_size
        self.lw = cfg.line_width
    
    def _get_filepath(self, idx: int) -> str:
        """Get filepath for given index."""
        if self.cfg.data_format == 'parquet':
            return os.path.join(self.cfg.data_dir, f'{self.cfg.file_prefix}{idx}.parquet')
        else:
            return os.path.join(self.cfg.data_dir, f'{self.cfg.file_prefix}{idx}.csv')
    
    def _read_file(self, filepath: str) -> pd.DataFrame:
        """Read file based on format."""
        if self.cfg.data_format == 'parquet':
            return pd.read_parquet(filepath)
        else:
            return pd.read_csv(filepath)
    
    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        
        file_indices = self.file_indices.copy()
        
        # Distribute files across workers
        if worker_info is not None:
            per_worker = len(file_indices) // worker_info.num_workers
            start = worker_info.id * per_worker
            if worker_info.id == worker_info.num_workers - 1:
                file_indices = file_indices[start:]
            else:
                file_indices = file_indices[start:start + per_worker]
        
        # Shuffle file order
        np.random.shuffle(file_indices)
        
        for idx in file_indices:
            filepath = self._get_filepath(idx)
            if not os.path.exists(filepath):
                continue
            
            df = self._read_file(filepath)
            df = df.sample(frac=1)  # Shuffle within file
            
            for _, row in df.iterrows():
                strokes = json.loads(row['drawing'])
                img = draw_strokes(strokes, size=self.size, lw=self.lw)
                img = torch.from_numpy(img).float().unsqueeze(0)
                img = (img / 127.5) - 1.0
                
                yield img, int(row['y'])

In [ ]:
# ============================================
# DATALOADER CREATION (FIXED)
# ============================================

def create_dataloaders(cfg: Config) -> Tuple[DataLoader, DataLoader, pd.DataFrame]:
    """Create train and validation dataloaders."""
    print("\nCreating datasets...")
    
    # Discover available files (now uses cfg for format info)
    available_indices, total_files = discover_data_files(cfg)
    
    if total_files == 0:
        raise FileNotFoundError(
            f"No preprocessed data found in {cfg.data_dir}\n"
            f"Expected format: {cfg.file_prefix}*.{cfg.data_format}\n"
            f"Please run: create_shuffled_csvs(CFG, force=True)"
        )
    
    # Split into train/valid
    train_indices, valid_indices = split_train_valid_indices(
        available_indices, 
        valid_ratio=cfg.valid_split
    )
    
    cfg.train_indices = train_indices
    cfg.valid_indices = valid_indices
    
    # Validation dataset
    valid_dataset = QuickDrawDataset(cfg, valid_indices, max_samples=34000)
    valid_df = valid_dataset.df.copy()
    
    # Training dataset
    train_dataset = QuickDrawStreamingDataset(cfg, train_indices)
    
    # Device-specific settings
    pin_memory = DEVICE_CFG.device_type == 'cuda'
    persistent_workers = cfg.num_workers > 0
    
    if DEVICE_CFG.device_type == 'mps':
        pin_memory = False
        persistent_workers = False
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.global_batch_size,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers and cfg.num_workers > 0
    )
    
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=cfg.global_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=pin_memory
    )
    
    print(f"\n✓ Dataloaders created ({cfg.data_format} format)")
    print(f"  Train: {len(train_indices)} files, batch_size={cfg.global_batch_size}")
    print(f"  Valid: {len(valid_dataset):,} samples")
    
    return train_loader, valid_loader, valid_df

## Part 4: Model Definition

In [ ]:
# ============================================
# MOBILENET MODEL
# ============================================

class QuickDrawMobileNet(nn.Module):
    """
    MobileNet-based model for Quick Draw classification.
    
    Supports both timm (optimized) and torchvision backends.
    Modified for single-channel (grayscale) input.
    """
    
    def __init__(self, num_classes: int = 340, pretrained: bool = False):
        super().__init__()
        
        if HAS_TIMM:
            # Use timm's optimized MobileNetV3
            self.backbone = timm.create_model(
                'mobilenetv3_large_100',
                pretrained=pretrained,
                in_chans=1,  # Grayscale
                num_classes=num_classes
            )
            self.model_name = 'MobileNetV3-Large (timm)'
        else:
            # Fallback to torchvision MobileNetV2
            self.backbone = mobilenet_v2(pretrained=False)
            
            # Modify first conv for single channel input
            old_conv = self.backbone.features[0][0]
            self.backbone.features[0][0] = nn.Conv2d(
                1, old_conv.out_channels,
                kernel_size=old_conv.kernel_size,
                stride=old_conv.stride,
                padding=old_conv.padding,
                bias=False
            )
            
            # Modify classifier
            self.backbone.classifier[1] = nn.Linear(
                self.backbone.classifier[1].in_features,
                num_classes
            )
            self.model_name = 'MobileNetV2 (torchvision)'
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


def create_model(cfg: Config) -> nn.Module:
    """
    Create and configure model for available device(s).
    
    Handles multi-GPU via DataParallel.
    """
    model = QuickDrawMobileNet(num_classes=cfg.num_classes)
    print(f"\nModel: {model.model_name}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Move to device
    model = model.to(DEVICE)
    
    # Multi-GPU wrapper
    if DEVICE_CFG.device_type == 'cuda' and DEVICE_CFG.num_devices > 1:
        model = nn.DataParallel(model)
        print(f"  Using DataParallel on {DEVICE_CFG.num_devices} GPUs")
    
    return model

## Part 5: Training Loop

In [ ]:
# ============================================
# METRICS
# ============================================

def top_k_accuracy(output: torch.Tensor, target: torch.Tensor, k: int = 3) -> float:
    """Calculate top-k accuracy."""
    with torch.no_grad():
        _, pred = output.topk(k, dim=1)
        correct = pred.eq(target.view(-1, 1).expand_as(pred))
        return correct.any(dim=1).float().mean().item()


def calculate_map3(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Calculate Mean Average Precision @ 3.
    
    Args:
        y_true: Ground truth labels (N,)
        y_pred: Prediction probabilities (N, num_classes)
    """
    top3_preds = np.argsort(-y_pred, axis=1)[:, :3]
    
    scores = []
    for true, preds in zip(y_true, top3_preds):
        score = 0.0
        for i, p in enumerate(preds):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    
    return np.mean(scores)

In [ ]:
# ============================================
# TRAINING & VALIDATION FUNCTIONS
# ============================================

def train_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer,
                scheduler: Optional[Any], scaler: Optional[GradScaler],
                steps_per_epoch: int) -> Dict[str, float]:
    """
    Train for one epoch.
    
    Returns:
        Dictionary with loss and accuracy metrics
    """
    model.train()
    
    total_loss = 0.0
    total_acc1 = 0.0
    total_acc3 = 0.0
    num_batches = 0
    
    pbar = tqdm(loader, total=steps_per_epoch, desc="Training")
    
    for batch_idx, (images, labels) in enumerate(pbar):
        if batch_idx >= steps_per_epoch:
            break
        
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        # Mixed precision forward pass
        if DEVICE_CFG.use_amp:
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = F.cross_entropy(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
        
        if scheduler is not None:
            scheduler.step()
        
        # Metrics
        with torch.no_grad():
            acc1 = (outputs.argmax(1) == labels).float().mean().item()
            acc3 = top_k_accuracy(outputs, labels, k=3)
        
        total_loss += loss.item()
        total_acc1 += acc1
        total_acc3 += acc3
        num_batches += 1
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'acc@1': f"{acc1:.3f}",
            'acc@3': f"{acc3:.3f}",
            'lr': f"{optimizer.param_groups[0]['lr']:.2e}"
        })
    
    return {
        'loss': total_loss / num_batches,
        'acc1': total_acc1 / num_batches,
        'acc3': total_acc3 / num_batches
    }


@torch.no_grad()
def validate(model: nn.Module, loader: DataLoader) -> Tuple[Dict[str, float], np.ndarray]:
    """
    Validate model on validation set.
    
    Returns:
        metrics: Dictionary with loss and accuracy
        predictions: Raw prediction probabilities for MAP@3
    """
    model.eval()
    
    total_loss = 0.0
    total_acc1 = 0.0
    total_acc3 = 0.0
    num_batches = 0
    
    all_preds = []
    
    for images, labels in tqdm(loader, desc="Validating"):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        
        if DEVICE_CFG.use_amp:
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = F.cross_entropy(outputs, labels)
        else:
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
        
        # Collect predictions
        probs = F.softmax(outputs.float(), dim=1)
        all_preds.append(probs.cpu().numpy())
        
        # Metrics
        acc1 = (outputs.argmax(1) == labels).float().mean().item()
        acc3 = top_k_accuracy(outputs, labels, k=3)
        
        total_loss += loss.item()
        total_acc1 += acc1
        total_acc3 += acc3
        num_batches += 1
    
    predictions = np.concatenate(all_preds, axis=0)
    
    return {
        'loss': total_loss / num_batches,
        'acc1': total_acc1 / num_batches,
        'acc3': total_acc3 / num_batches
    }, predictions

In [ ]:
# ============================================
# MAIN TRAINING LOOP
# ============================================

def train_model(cfg: Config, steps_per_epoch: int = 800) -> Tuple[nn.Module, Dict]:
    """
    Full training pipeline.
    
    Returns:
        model: Trained model
        history: Training history
    """
    print("\n" + "="*60)
    print("TRAINING SETUP")
    print("="*60)
    
    # Create data
    train_loader, valid_loader, valid_df = create_dataloaders(cfg)
    
    # Create model
    model = create_model(cfg)
    
    # Optimizer
    optimizer = AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )
    
    # Scheduler: OneCycleLR for fast convergence
    total_steps = steps_per_epoch * cfg.epochs
    scheduler = OneCycleLR(
        optimizer,
        max_lr=cfg.lr,
        total_steps=total_steps,
        pct_start=cfg.warmup_epochs / cfg.epochs,
        anneal_strategy='cos'
    )
    
    # Mixed precision scaler
    scaler = torch.amp.GradScaler('cuda') if DEVICE_CFG.use_amp else None
    
    # Training state
    history = {
        'train_loss': [], 'train_acc1': [], 'train_acc3': [],
        'valid_loss': [], 'valid_acc1': [], 'valid_acc3': [], 'valid_map3': []
    }
    best_map3 = 0.0
    patience_counter = 0
    patience = 999
    
    print(f"\n{'='*60}")
    print(f"TRAINING")
    print(f"{'='*60}")
    print(f"Epochs: {cfg.epochs}")
    print(f"Steps/epoch: {steps_per_epoch}")
    print(f"Total steps: {total_steps:,}")
    print(f"Device: {DEVICE_CFG.device_type.upper()} x{DEVICE_CFG.num_devices}")
    print(f"AMP: {DEVICE_CFG.use_amp}")
    print("="*60 + "\n")
    
    train_start = dt.datetime.now()
    
    for epoch in range(cfg.epochs):
        epoch_start = dt.datetime.now()
        print(f"\nEpoch {epoch+1}/{cfg.epochs}")
        print("-" * 40)
        
        # Train
        train_metrics = train_epoch(
            model, train_loader, optimizer, scheduler, scaler, steps_per_epoch
        )
        
        # Validate
        valid_metrics, valid_preds = validate(model, valid_loader)
        
        # Calculate MAP@3
        map3 = calculate_map3(valid_df['y'].values, valid_preds)
        valid_metrics['map3'] = map3
        
        # Update history
        for k, v in train_metrics.items():
            history[f'train_{k}'].append(v)
        for k, v in valid_metrics.items():
            history[f'valid_{k}'].append(v)
        
        epoch_time = (dt.datetime.now() - epoch_start).seconds
        
        print(f"\n  Train - Loss: {train_metrics['loss']:.4f}, "
              f"Acc@1: {train_metrics['acc1']:.4f}, Acc@3: {train_metrics['acc3']:.4f}")
        print(f"  Valid - Loss: {valid_metrics['loss']:.4f}, "
              f"Acc@1: {valid_metrics['acc1']:.4f}, Acc@3: {valid_metrics['acc3']:.4f}")
        print(f"  MAP@3: {map3:.4f}  |  Time: {epoch_time}s")
        
        # Save best model
        if map3 > best_map3:
            best_map3 = map3
            patience_counter = 0
            
            # Save model
            state_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            torch.save(state_dict, '/kaggle/working/best_model.pt')
            print(f"  ✓ New best model saved! MAP@3: {best_map3:.4f}")
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{patience})")
        
        # Early stopping
        if patience_counter >= patience:
            print(f"\n⚠ Early stopping triggered at epoch {epoch+1}")
            break
        
        # Clear cache
        if DEVICE_CFG.device_type == 'cuda':
            torch.cuda.empty_cache()
    
    train_time = (dt.datetime.now() - train_start).total_seconds()
    print(f"\n✓ Training completed in {train_time/3600:.2f} hours")
    print(f"  Best MAP@3: {best_map3:.4f}")
    
    # Load best weights
    if os.path.exists('/kaggle/working/best_model.pt'):
        state_dict = torch.load('/kaggle/working/best_model.pt', map_location=DEVICE)
        if hasattr(model, 'module'):
            model.module.load_state_dict(state_dict)
        else:
            model.load_state_dict(state_dict)
        print("  ✓ Loaded best model weights")
    
    return model, history

In [ ]:
# ============================================
# RUN TRAINING
# ============================================

model, history = train_model(CFG, steps_per_epoch=800)

In [ ]:
# ============================================
# PLOT TRAINING HISTORY
# ============================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train', lw=2)
axes[0, 0].plot(history['valid_loss'], label='Valid', lw=2)
axes[0, 0].set_title('Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Top-1 Accuracy
axes[0, 1].plot(history['train_acc1'], label='Train', lw=2)
axes[0, 1].plot(history['valid_acc1'], label='Valid', lw=2)
axes[0, 1].set_title('Top-1 Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Top-3 Accuracy
axes[1, 0].plot(history['train_acc3'], label='Train', lw=2)
axes[1, 0].plot(history['valid_acc3'], label='Valid', lw=2)
axes[1, 0].set_title('Top-3 Accuracy')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# MAP@3
axes[1, 1].plot(history['valid_map3'], label='Valid MAP@3', lw=2, color='green')
axes[1, 1].set_title('MAP@3')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

print(f"\nBest MAP@3: {max(history['valid_map3']):.4f}")

## Part 6: Generate Submission

In [ ]:
# ============================================
# GENERATE SUBMISSION
# ============================================

@torch.no_grad()
def generate_submission(model: nn.Module, cfg: Config) -> str:
    """
    Generate competition submission file.
    
    Returns:
        Path to saved submission file
    """
    model.eval()
    
    print("\nGenerating submission...")
    
    # Load test data
    test_df = pd.read_csv(os.path.join(cfg.input_dir, 'test_simplified.csv'))
    print(f"  Test samples: {len(test_df):,}")
    
    # Create test dataset
    class TestDataset(Dataset):
        def __init__(self, df, size, lw):
            self.df = df
            self.size = size
            self.lw = lw
        
        def __len__(self):
            return len(self.df)
        
        def __getitem__(self, idx):
            strokes = json.loads(self.df.iloc[idx]['drawing'])
            img = draw_strokes(strokes, size=self.size, lw=self.lw)
            img = torch.from_numpy(img).float().unsqueeze(0)
            img = (img / 127.5) - 1.0
            return img
    
    test_dataset = TestDataset(test_df, cfg.image_size, cfg.line_width)
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.global_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(DEVICE_CFG.device_type == 'cuda')
    )
    
    # Predict
    all_preds = []
    for images in tqdm(test_loader, desc="Predicting"):
        images = images.to(DEVICE, non_blocking=True)
        
        if DEVICE_CFG.use_amp:
            with torch.amp.autocast('cuda'):
                outputs = model(images)
        else:
            outputs = model(images)
        
        # Get top 3 predictions
        top3 = outputs.topk(3, dim=1).indices.cpu().numpy()
        all_preds.append(top3)
    
    predictions = np.concatenate(all_preds, axis=0)
    
    # Map to category names
    categories = list_all_categories(cfg.input_dir)
    id2cat = {i: cat.replace(' ', '_') for i, cat in enumerate(categories)}
    
    # Create submission
    words = []
    for pred in predictions:
        word = ' '.join([id2cat[p] for p in pred])
        words.append(word)
    
    test_df['word'] = words
    
    # Save
    best_map3 = max(history['valid_map3'])
    filename = f'submission_map3_{int(best_map3 * 10000)}.csv'
    test_df[['key_id', 'word']].to_csv(filename, index=False)
    
    print(f"\n✓ Submission saved: {filename}")
    print(f"\nPreview:")
    print(test_df[['key_id', 'word']].head(10))
    
    return filename


submission_file = generate_submission(model, CFG)

In [ ]:
# ============================================
# FINAL SUMMARY
# ============================================

end_time = dt.datetime.now()
total_time = (end_time - START_TIME).total_seconds()

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\nDevice: {DEVICE_CFG.device_type.upper()}")
print(f"  Count: {DEVICE_CFG.num_devices}")
print(f"  Names: {', '.join(DEVICE_CFG.device_names)}")
print(f"  AMP: {'Enabled' if DEVICE_CFG.use_amp else 'Disabled'}")
print(f"\nData:")
print(f"  Samples per class: {CFG.samples_per_class:,}")
print(f"  Image size: {CFG.image_size}x{CFG.image_size}")
print(f"\nResults:")
print(f"  Best MAP@3: {max(history['valid_map3']):.4f}")
print(f"  Best Top-3 Acc: {max(history['valid_acc3']):.4f}")
print(f"\nTime:")
print(f"  Total: {total_time/3600:.2f} hours ({total_time/60:.1f} minutes)")
print(f"  Started: {START_TIME}")
print(f"  Finished: {end_time}")
print("="*60)